<div style="font-size: 40px; font-weight: bold;">Initialization</div>


In [1]:
%run basic_initialization.ipynb
plotFigs = False
saveFigs = False


Fri 02/10/26 00:14:46


In [2]:

model_name = '3-16-8-1'
activations =['sigmoid','tanh', 'relu', 'softplus', 'swish', 'exponential']
Datafile = 'trained_models'
Fortran_folder = 'trained_fortran'

Adddata_h = True
Adddata_u = True
Explicit = True
useData = True

inlineFactor = True




In [22]:
# fortran block
comment = 'C '
cline = 'C **********************************************************************\n'
block =   '      '   
variableFormatToUse = "%.16f"

def valueString(a):
    s = variableFormatToUse%(a)
    for c in range(len(s)-1,0,-1):
        if (s[c] != '0'): break
    if (s[c]=='.'): return s[0:c]
    return s[0:c+1]

# basic fortran setting
def toFortran(line):
    line = line.replace("--", "+")
    line = line.replace("max", "dmax1")
    line = line.replace("+-", "-")
    line = line.replace("(+", "(")
    line = line.replace('+',' + ')
    line = line.replace('-',' - ')
    line = line.replace('=  - ','=-')
    line = line.replace('( - ','(-')
    line = line.replace('  +  ',' + ')
    line = line.replace('.000000000000','.0')
    if (line[0] == 'C'): line = line + '\n'
    else: line = block + line + '\n'
    line = splitFortranLine(line)
    return line


def splitFortranLine(line):
    line = line.replace('|*','\n     + *')
    line = line.replace('|/','\n     + /')
    line = line.replace('| + ','\n     + +')
    line = line.replace('| - ','\n     + -')
    return line

# vuhard head
def vuhardHead(activation, Adddata_h,w,b,r):
    code = cline
    code += comment + 'Function to compute the ANN : ' + Datafile + '-' + model_name + activation+' yield stress\n'
    code += cline
    code += '      subroutine vuhard (\n'
    code += 'C Read only -\n'
    code += '     +  nblock, nElement, nIntPt, nLayer, nSecPt, lAnneal, stepTime,\n'
    code += '     +  totalTime, dt, cmname, nstatev, nfieldv, nprops, props,\n'
    code += '     +  tempOld, tempNew, fieldOld, fieldNew, stateOld, eqps, eqpsRate,\n'
    code += 'C Write only -\n'
    code += '     +  yield, dyieldDtemp, dyieldDeqps, stateNew)\n'
    code += 'C\n'
    code += "      include 'vaba_param.inc'\n"
    code += 'C\n'
    code += '      dimension nElement(nblock), props(nprops), tempOld(nblock),\n'
    code += '     +  fieldOld(nblock,nfieldv), stateOld(nblock,nstatev),\n'
    code += '     +  tempNew(nblock), fieldNew(nblock,nfieldv), eqps(nblock),\n'
    code += '     +  eqpsRate(nblock), yield(nblock), dyieldDtemp(nblock),\n'
    code += '     +  dyieldDeqps(nblock,2), stateNew(nblock,nstatev)\n'
    code += 'C\n'
    code += '      character*80 cmname\n'
    if (Adddata_h):
        code = dataZone(code,w,b,r)
    code += 'C Do the main loop for all block values\n'
    code += '      do k = 1, nblock\n'
    return code

# uhard head
def uhardHead(activation,Adddata_u,w,b,r):
    code = cline
    code += comment + 'Function to compute the ANN : ' + Datafile + '-' + model_name + activation+' yield stress\n'
    code += cline
    code += '      subroutine uhard (\n'
    code += '     +  syield, hard, eqplas, eqplasrt, time, dtime, temp, \n'
    code += '     +  dtemp, noel, npt, layer, kspt, kstep, kinc, \n'
    code += '     +  cmname, nstatv, statev, numfieldv, \n'
    code += '     +  predef, dpredef, numprops, props)\n'
    code += 'C\n'
    code += "      include 'aba_param.inc'\n"
    code += 'C\n'
    code += '      dimension hard(3), statev(nstatv), time(*),\n'
    code += '     +  predef(numfieldv), dpredef(*), props(*)\n'
    code += 'C\n'
    code += '      character*80 cmname\n'
    if (Adddata_u):
        code = dataZone(code,w,b,r)
    return code

# fortran tail
def vuhardTail():
    code = 'C Store the eqpsRate into stateNew variable 1\n'
    code += '      stateNew(k,1) = eqpsRate(k)\n'
    code += '      end do\n'
    code += 'C Return from the VUHARD subroutine\n'
    code += '      return\n'
    code += '      end\n'
    return code

def uhardTail():
    code = 'C Store the eqplasrt into statev variable 1\n'
    code += '      statev(1) = eqplasrt\n'
    code += 'C Return from the UHARD subroutine\n'
    code += '      return\n'
    code += '      end\n'
    return code

# data part
def dataZone(code,w,b,r):
    code += 'C Block of Data\n'
    code += writeDataFortran(w[0], 'w1')
    code += writeDataFortran(b[0], 'b1')
    code += writeDataFortran(w[1], 'w2')
    code += writeDataFortran(b[1], 'b2')
    code += writeDataFortran(w[2], 'w3')
    code += writeDataFortran(b[2], 'b3')
    code += writeDataFortran(r[0][0:3], 'xmI')
    code += writeDataFortran(r[1][0:3]-r[0][0:3], 'xrI')
    code += writeDataFortran(r[0][3:4], 'xmO')
    code += writeDataFortran(r[1][3:4]-r[0][3:4], 'xrO')
    code += writeDataFortran(r[2].reshape(1, 1), 'xdeps0')
    return code


def writeDataFortran(var, name):
    I, J = var.shape
    lines = ''
    if (I == 1) and (J==1):
        lines += toFortran('double precision '+name)
    elif (I == 1) or (J==1):
        if (I == 1):
            lines += toFortran('double precision '+name+'('+str(J)+')')
        if (J == 1):
            lines += toFortran('double precision '+name+'('+str(I)+')')
    else:
        lines += toFortran('double precision '+name+'('+str(I)+', '+str(J)+')')
    line = '      data '+name+'/'
    d = 0
    for j in range (J):
        for i in range (I):
            if (d>0): line += ',\n     + '
            line += valueString(var[i,j])+'D0'
            d += 1
    line += '/\n'        
    lines += line
    return lines


#calculation part

# activation function total 6 function 6 derivation
def actFortran(activation, x, i):
    exist = False
    if (activation == 'sigmoid'):
        exist = True
        fct = '1/(1+exp(-X))'
    if (activation == 'exponential'):
        exist = True
        fct = 'exp(X)'
    if (activation == 'tanh'):
        exist = True
        fct = 'tanh(X)'
    if (activation == 'swish'):
        exist = True
        fct = 'X/(1+exp(-X))'
    if (activation == 'softplus'):
        exist = True
        #fct = 'log(1+exp(-abs(X)))+max(0.0D0,X)'
        fct = 'log(1+exp(X))'
        # np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)
    if (activation == 'relu'):
        exist = True
        fct = '(max(0.0D0,X))'
        # np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)
    if (not exist):
        print('ERROR Its not implemented yet')
        return '',''
    return fct.replace('X',x).replace('#',str(i))

def dactFortran(activation, y, x, i):
    exist = False
    if (activation == 'sigmoid'):
        exist = True
        fct = '(Y*(1-Y))' # sigmoid(x)*(1-sigmoid(x)) = y*(1-y)
    if (activation == 'exponential'):
        exist = True
        fct = 'Y' # exp(x) = y
    if (activation == 'tanh'):
        exist = True
        fct = '(1-Y*Y)' # 1-tanh(x)**2 = 1-y*y
    if (activation == 'swish'):
        exist = True
        fct = '(Y+(1-Y)/(1+exp(-X)))' # swish(x) + sigmoid(x)*(1 - swish(x)) = y + sigmoid(x)*(1-y)
    if (activation == 'softplus'):
        exist = True
        fct = '(1/(1+exp(-X)))' # swish(x) + sigmoid(x)*(1 - swish(x)) = y + sigmoid(x)*(1-y)
    if (activation == 'relu'):
        exist = True
        fct = 'max(0.0D0,sign(1.0D0,X))'
    if (not exist):
        print('ERROR Its not implemented yet')
        return '',''
    return fct.replace('X',x).replace('Y',y).replace('#',str(i))



# calculation part
def encodeFunction(f,func):
    f += toFortran(func)
    return f

def process_var_line(fortranCode):
    fortranCode = encodeFunction(fortranCode, comment + 'Preprocessing of the variables')
    return fortranCode

if (Explicit):
    var_xeps = 'eqps(k)'
    var_xdeps = 'eqpsRate(k)'
    var_xtemp = 'tempNew(k)'
else:
    var_xeps = 'eqplas'
    var_xdeps = 'eqplasrt'
    var_xtemp = 'temp+dtemp'

def preprocess_var(fortranCode):
    fortranCode = process_var_line(fortranCode)


    if (useData):
        fortranCode = encodeFunction(fortranCode,'xeps = ('+var_xeps+'-xmI(1))/xrI(1)')
    else:
        fortranCode= encodeFunction(fortranCode, 'xeps = ('+var_xeps+'-' + valueString(minInput[0,0]) + ')/' + valueString(rangeInput[0,0]))

    # Compute for deps
    if (useData):
        fortranCode = encodeFunction(fortranCode, 'if ('+var_xdeps+' > xdeps0) then')
        fortranCode = encodeFunction(fortranCode, '  xdeps = (log('+var_xdeps+'/xdeps0)-xmI(2))/xrI(2)')
        fortranCode = encodeFunction(fortranCode, 'else')
        fortranCode = encodeFunction(fortranCode, '  xdeps = 0')
        fortranCode = encodeFunction(fortranCode, '  '+var_xdeps+' = xdeps0')
        fortranCode = encodeFunction(fortranCode, 'endif')
    else:
        fortranCode = encodeFunction(fortranCode, 'if ('+var_xdeps+' > ' + valueString(deps0) + ') then')
        fortranCode = encodeFunction(fortranCode, '  xdeps = (log('+var_xdeps+'/' + valueString(deps0) + ')-' + valueString(minInput[1,0]) + ')/' + valueString(rangeInput[1,0]))
        fortranCode = encodeFunction(fortranCode, 'else')
        fortranCode = encodeFunction(fortranCode, '  xdeps = 0')
        fortranCode= encodeFunction(fortranCode,  '  '+var_xdeps+' = ' + valueString(deps0))
        fortranCode = encodeFunction(fortranCode, 'endif')

    # Compute for T
    if (useData):
        fortranCode = encodeFunction(fortranCode, 'xtemp = ('+var_xtemp+'-xmI(3))/xrI(3)')
    else:
        fortranCode = encodeFunction(fortranCode, 'xtemp = ('+var_xtemp+'-' + valueString(minInput[2,0])+')/' + valueString(rangeInput[2,0]))
    return fortranCode

def wv1(n,i):
    return n+'('+str(i+1)+')'
    
def wv2(n,i,j):
    return n+'('+str(i+1)+','+str(j+1)+')'

def wv(i):
    return str(i+1)    


def calculation_layer1(w,b,r, activation,fortranCode):
    w1 , b1 = w[0] , b[1]
    fortranCode= encodeFunction(fortranCode,comment + 'Hidden layer #1-(y11 to y1'+str(b1.shape[0])+')')
    # Write the function
    for i in range(b1.shape[0]):
        if (useData):
            fortranCode = encodeFunction(fortranCode, 'y1' + wv(i) + ' = ' + wv2('w1',i,0) + ' * xeps|+' + wv2('w1',i,1) + ' * xdeps|+' + wv2('w1',i,2) + ' * xtemp|+' + wv1('b1',i))
        else:
            fortranCode = encodeFunction(fortranCode,'y1' + wv(i) + ' = ' + valueString(w1[i,0]) + ' * xeps|+'+ valueString(w1[i,1]) + ' * xdeps|+'+ valueString(w1[i,2]) + ' * xtemp|+' + valueString(b1[i,0]))

    fortranCode = encodeFunction(fortranCode, comment + activation + ' activation function-(yf11 to yf1'+str(b1.shape[0])+')')

    for i in range(b1.shape[0]):
        fortranCode = encodeFunction(fortranCode,  'yf1' + wv(i) + ' = ' + actFortran(activation, 'y1#', i+1))

    return fortranCode

    


def calculation_layer2(w,b,r, activation,fortranCode):
    w2 , b2 = w[1] , b[2]
    # Write the comment for the function
    fortranCode= encodeFunction(fortranCode, comment + 'Hidden layer #2-(y21 to y2'+str(b2.shape[0])+')')
    # Write the function
    if (useData):
        for i in range(b2.shape[0]):
            line = 'y2' + wv(i) + " = "
            for j in range(w2.shape[1]):
                if (j != 0): line += '+'
                line += wv2('w2',i,j) + ' * yf1' + wv(j) + '|'
            line += '+' + wv1('b2',i)
            fortranCode = encodeFunction(fortranCode,line)
    else:
        for i in range(b2.shape[0]):
            line = 'y2' + wv(i) + " = "
            for j in range(w2.shape[1]):
                line += '+' + valueString(w2[i,j]) + ' * yf1' + wv(j) + '|'
            line += '+' + valueString(b2[i,0])
            fortranCode = encodeFunction(fortranCode, line)

    fortranCode = encodeFunction(fortranCode, comment + activation + ' activation function-(yf21 to yf2'+str(b2.shape[0])+')')

    for i in range(b2.shape[0]):
        fortranCode = encodeFunction(fortranCode,'yf2' + wv(i) + ' = ' + actFortran(activation, 'y2#', i+1))

    return fortranCode
   

def calculation_layer3(w,b,r, activation,fortranCode):
    w3 = w[2]; b3= b[2]; w2 = w[1] 
    fortranCode = encodeFunction(fortranCode, comment + 'Derivatives terms-(xa1 to xa'+str(w3.shape[1])+') and (xb1 to xb'+str(w2.shape[1])+')')
    # xa = w3.T * dactivation(y2))
    if (useData):
        for i in range(w3.shape[1]):
            fortranCode = encodeFunction(fortranCode,'xa' + wv(i) + ' = ' + wv1('w3',i) + ' * ' + dactFortran(activation,'yf2#','y2#',i+1))
    else:
        for i in range(w3.shape[1]):
            fortranCode = encodeFunction(fortranCode,'xa' + wv(i) + ' = ' + valueString(w3[0,i]) + ' * ' + dactFortran(activation,'yf2#','y2#',i+1))

    # xb = (w2.T).dot(xa * dactivation(y1)
    if (useData):    
        for i in range(w2.shape[1]):
            line = 'xb' + wv(i) + ' = ('
            for j in range(w2.shape[0]):
                if (j!=0): line += '|+'
                line += wv2('w2',j,i) + ' * ' + 'xa' + wv(j)
            line += ')|* ' + dactFortran(activation,'yf1#','y1#',i+1) #yf1' + wv(i) + '*(1-yf1' + wv(i) + ')'
            fortranCode = encodeFunction(fortranCode, line)
    else:
        for i in range(w2.shape[1]):
            line = 'xb' + wv(i) + ' = ('
            for j in range(w2.shape[0]):
                if (j!=0): line += '|+'
                line += valueString(w2[j,i]) + ' * ' + 'xa' + wv(j)
            line += ')|* ' + dactFortran(activation,'yf1#','y1#',i+1) #yf1' + wv(i) + '*(1-yf1' + wv(i) + ')'
    fortranCode= encodeFunction(fortranCode,line)
    return fortranCode



if (Explicit):
    var_sig = 'Yield(k)'
    var_dsig1 = 'dyieldDeqps(k,1)'
    var_dsig2 = 'dyieldDeqps(k,2)'
    var_dsig3 = 'dyieldDtemp(k)'
else:
    var_sig = 'syield'
    var_dsig1 = 'hard(1)'
    var_dsig2 = 'hard(2)'
    var_dsig3 = 'hard(3)'


def calculation_result(fortranCode,activation,w,b,r):
    # Write the comment for the function
    fortranCode = encodeFunction(fortranCode, comment + 'Outputs of the subroutine')
    w , b , r =  ReadANNH5(f"{Datafile}/{model_name}-{activation}-ANN.h5")
    w3 = w[2];b3 = b[2];  w1 = w[0]
    #Write the function
    if (useData):    
        line = var_sig + ' = xrO * ('
        for i in range(w3.shape[1]):
            if (i != 0): line += '|+'
            line += wv1('w3',i) + ' * yf2' + wv(i)
        line += '|+b3)|+xmO' 
        fortranCode = encodeFunction(fortranCode,line)
    else:
        if (inlineFactor):
            line = var_sig + ' = '
            for i in range(w3.shape[1]):
                if (i != 0): line += '|+'
                line += valueString(rangeOutput[0] * w3[0,i]) + ' * yf2' + wv(i)
            line += '|+' + valueString(rangeOutput[0] * b3[0][0] + minOutput[0]) 
            fortranCode = encodeFunction(fortranCode,  line)
        else:
            line = var_sig + ' = ' + valueString(rangeOutput[0]) + '|*('
            for i in range(w3.shape[1]):
                if (i != 0): line += '|+'
                line += valueString(w3[0,i]) + ' * yf2' + wv(i)
            line += '|+' + valueString(b3[0][0]) + ')|+' + valueString(minOutput[0]) 
            fortranCode = encodeFunction(fortranCode,line)
    
    # Write the derivatives of the functions
    if (useData):   
        line = var_dsig1 + ' = xrO * ('
        for j in range(w1.shape[0]):
            if (j != 0): line += '|+'
            line += wv2('w1',j,0) + ' * ' + 'xb' + wv(j)
        line += ') / xrI(1)'
        fortranCode = encodeFunction(fortranCode, line)
        line = var_dsig2 + ' = xrO * ('
        for j in range(w1.shape[0]):
            if (j != 0): line += '|+'
            line += wv2('w1',j,1) + ' * ' + 'xb' + wv(j)
        line += ')|/(xrI(2)*'+var_xdeps+')'
        fortranCode = encodeFunction(fortranCode, line)
        line = var_dsig3 + ' = xrO * ('
        for j in range(w1.shape[0]):
            if (j != 0): line += '|+'
            line += wv2('w1',j,2) + ' * ' + 'xb' + wv(j)
        line += ') / xrI(3)'
        fortranCode = encodeFunction(fortranCode,  line)
    else:
        if (inlineFactor):
            line = var_dsig1 + ' = '
            for j in range(w1.shape[0]):
                if (j != 0): line += '|+'
                line += valueString(rangeOutput[0]*w1[j,0]/rangeInput[0,0]) + ' * ' + 'xb' + wv(j)
            fortranCode = encodeFunction(fortranCode,  line)
            line = var_dsig2 + ' = ('
            for j in range(w1.shape[0]):
                if (j != 0): line += '|+'
                line += valueString(rangeOutput[0]*w1[j,1]/rangeInput[1,0]) + ' * ' + 'xb' + wv(j)
            line += ') / '+var_xdeps+''
            fortranCode = encodeFunction(fortranCode,  line)
            line = var_dsig3 + ' = '
            for j in range(w1.shape[0]):
                if (j != 0): line += '|+'
                line += valueString(rangeOutput[0]*w1[j,2]/rangeInput[2,0]) + ' * ' + 'xb' + wv(j)
            fortranCode= encodeFunction(fortranCode, line)
        else:
            line = var_dsig1 + ' = ' + valueString(rangeOutput[0]) + '|* ('
            for j in range(w1.shape[0]):
                if (j != 0): line += '|+'
                line += valueString(w1[j,0]) + ' * ' + 'xb' + wv(j)
            line += ') / ' + valueString(rangeInput[0,0])
            fortranCode = encodeFunction(fortranCode, line)
            line = var_dsig2 + ' = ' + valueString(rangeOutput[0]) + '|* ('
            for j in range(w1.shape[0]):
                if (j != 0): line += '|+'
                line += valueString(w1[j,1]) + ' * ' + 'xb' + wv(j)
            line += ')|/(' + valueString(rangeInput[1,0]) + '*'+var_xdeps+')'
            fortranCode = encodeFunction(fortranCode,  line)
            line = var_dsig3 + ' = ' + valueString(rangeOutput[0]) + '|* ('
            for j in range(w1.shape[0]):
                if (j != 0): line += '|+'
                line += valueString(w1[j,2]) + ' * ' + 'xb' + wv(j)
            line += ') / ' + valueString(rangeInput[2,0])
            fortranCode = encodeFunction(fortranCode, line)
    return fortranCode   





# calculation
def calculation_var(fortranCode,activation,w,b,r):
    fortranCode = preprocess_var(fortranCode)
    fortranCode = calculation_layer1(w,b,r, activation,fortranCode)
    fortranCode = calculation_layer2(w,b,r, activation,fortranCode)
    fortranCode = calculation_layer3(w,b,r, activation,fortranCode)
    fortranCode = calculation_result(fortranCode,activation,w,b,r)
    return fortranCode








## Fortran code generation method

In [ ]:

def generate_fortran_code(activation, Datafile):
    fortranCode = ''
    w , b , r =  ReadANNH5(f"{Datafile}/{model_name}-{activation}-ANN.h5")
    if (Explicit):
        fortranCode += vuhardHead(activation, Adddata_h,w,b,r)
    else:
        fortranCode += uhardHead(factivation, Adddata_v,w,b,r)
    fortranCode = preprocess_var(fortranCode)
    fortranCode = calculation_var(fortranCode,activation,w,b,r)
    if (Explicit):
        fortranCode += vuhardTail()
    else:
        fortranCode += uhardTail()    
    return fortranCode





3
3
C **********************************************************************
C Function to compute the ANN : trained_models-3-16-8-1relu yield stress
C **********************************************************************
      subroutine vuhard (
C Read only -
     +  nblock, nElement, nIntPt, nLayer, nSecPt, lAnneal, stepTime,
     +  totalTime, dt, cmname, nstatev, nfieldv, nprops, props,
     +  tempOld, tempNew, fieldOld, fieldNew, stateOld, eqps, eqpsRate,
C Write only -
     +  yield, dyieldDtemp, dyieldDeqps, stateNew)
C
      include 'vaba_param.inc'
C
      dimension nElement(nblock), props(nprops), tempOld(nblock),
     +  fieldOld(nblock,nfieldv), stateOld(nblock,nstatev),
     +  tempNew(nblock), fieldNew(nblock,nfieldv), eqps(nblock),
     +  eqpsRate(nblock), yield(nblock), dyieldDtemp(nblock),
     +  dyieldDeqps(nblock,2), stateNew(nblock,nstatev)
C
      character*80 cmname
C Block of Data
      double precision w1(16, 3)
      data w1/-0.363210312200735D0,
     + 0.

## Fortran file generation method

In [25]:

os.makedirs(Fortran_folder, exist_ok=True)


def generate_fortran_file(fortran_code, name):
    path = f"{Fortran_folder}/{name}.f"
    with open(path, "w") as text_file:
        text_file.write(fortran_code)
        text_file.close()

# call:



## Writes the fortran file

In [26]:

for activation in activations:
    fortran_code = generate_fortran_code(activation,Datafile)
    generate_fortran_file(fortran_code, f"{model_name}-{activation}")




3
3
3
3
3
3
3
3
3
3
3
3
